In [2]:
import pandas as pd
import numpy as np
import openpyxl as xl

In [4]:
path = "../raw-data/SHPD consolidated.xlsx"
det_df = pd.read_excel(path, sheet_name = "21-24 Det")

In [5]:
det_df.describe()

,Project Number,Project Name,Date\nOpinion 106,Opinion 106,Opinion 106\nComment,Date Opinion 6E,Opinion 6E,Opinion 6E Comment,Statutes,Agency,Island,Tax Map Keys
count,2604,2604,326,318,66,2423,2420,638,2604,2490,2502,2549
unique,2585,2594,261,6,58,858,11,464,32,258,12,2420
top,2020PR35019,County of Maui Permit Application B T2020/029 ...,#########,No Historic Properties Affected,NHPA,#########,No Historic Properties Affected,NHPA,HRS 6E-42,"County of Hawaii, Department of Public Works",Oʻahu,(1) 2-8-023:003
freq,2,2,10,227,3,98,2127,104,1648,220,760,12


In [9]:
print(det_df["Opinion 6E"].unique())

# so there is no effect, effect with proposal, and effect with agreement

<StringArray>
[                 'No Historic Properties Affected',
     'Effect, with Proposed Mitigation Commitme nt',
                                                nan,
 'Effect, with Agreed upon Mitigation Commitme nts',
  'Effect, with Agreed upon Mitigation Commitments',
     'Effect, with Proposed Mitigation Commitments',
 'Effect, with Agreed upon Mitigation\nCommitments',
                 'No Historic\nProperties Affected',
    'Effect, with Proposed Mitigation\nCommitments',
    'Effect, with\nProposed Mitigation Commitments',
    'Effect, with Proposed\nMitigation Commitments',
 'Effect, with Agreed\nupon Mitigation Commitments']
Length: 12, dtype: str


## Cleaning

We have to make some things right for the analysis.

1. need to create year variable using substring from permit id
2. need to standardize formats (use the clean part of the workbook?)

In [12]:
# cleaning up the dependent output variables

def classify_outcome(text: str) -> str:
    # if it does not exist
    if not isinstance(text, str):
        return None
    # forces to be lower case and without any spaces/newlines
    s = text.strip().lower()

    if "no" in s:
        return "no effect"
    elif "agreed" in s:
        return "effect with agreed commitments"
    elif "proposed" in s:
        return "effect with proposed commitments"
    else:
        return "NA"

def binary_outcome(text: str) -> str:
    if not isinstance(text, str):
        return None
    
    s = text.strip().lower() #lower just in case
    if "no" in s:
        return "no effect"
    if "effect" in s:
        return "effect"
    else:
        return "NA"


det_df["Opinion 6E"] = det_df["Opinion 6E"].apply(classify_outcome)
print(det_df["Opinion 6E"].unique())

det_df["opinion_6e_binary"] = det_df["Opinion 6E"].apply(binary_outcome)

<StringArray>
[                       'no effect', 'effect with proposed commitments',
                                nan,   'effect with agreed commitments']
Length: 4, dtype: str


In [ ]:
# cleaning descriptions
from spellchecker import SpellChecker
import re

stopwords = {
    "permit", "permits", "building", "department",
    "project", "work", "construction",
    "application", "owner", "existing", 
    "hawaii", "dpp", "maui", "oahu", 
    "kauai", "molokai", "honolulu"
}

def clean(text, stopwords = stopwords):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

    #remove matches to stopwords
    pattern = r'\b(' + '|'.join(re.escape(word) for word in stopwords) + r')\b\s*'
    text = re.sub(pattern, '', text)
    return text.strip()


det_df["Project Name"] = det_df["Project Name"].apply(clean)

spell = SpellChecker()

def spellCheck(text: str) -> str:
    if pd.isna(text):
        return text
    return " ".join(spell.correction(w) or w for w in text.split())

det_df["Project Name"] = det_df["Project Name"].apply(spellCheck)


KeyboardInterrupt: 

In [62]:
for name in det_df["Project Name"]:
    if "permit" in name:
        print(name)

dpp permits for matthew  alicia willis permit fax grading permit for access  pad permit additionsa lterations to ex building
county of hawaii grading permit for james gannon mgr 

mamalahoa hwy kailua kona hawaii
county of maui permit application b t  b t
 and b t
farm dwelling pool and retaining walls
project
county of maui permit application b t  b t
 and b t
farm dwelling pool and retaining walls
project
county of hawaii grading permit for cmc
partners kaloko llc 
maiau street kailua kona hawaii
special manageme nt area sma
assessment application  smx 
gradin g permit application  g t
buildin g permit application  b t 
county of maui permit application wtp t
count y of maui departmen t of water supply dws job no 
upcount y phase  pump station upgrades project
request for comments water use permit application mokuleia ground water manageme nt area mokuleia oahu
building permits  b t 
lahaole place  proposed additions to existing dwelling
county of maui permit application wtp t
 water

# Unsupervised Learning - K Means Clustering

Typical, but it is a start.


In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# we vectorize
desc = det_df["Project Name"]

# tf-idf vectorizer
vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df = 5,
    ngram_range=(1,2))

X = vectorizer.fit_transform(desc)

# then we run
kmeans = KMeans(n_clusters=5, random_state=0)
labels = kmeans.fit_predict(X)

det_df["cluster"] = labels
# still in numbers so have to interpret them

In [52]:
terms = vectorizer.get_feature_names_out()
centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for i in range(5):
    print("Cluster", i)
    print([terms[ind] for ind in centroids[i, :5]])

Cluster 0
['gd', 'pweng', 'water', 'improvements', 'subdivision']
Cluster 1
['permit', 'permit application', 'application', 'grading permit', 'coh']
Cluster 2
['building', 'building permit', 'permit', 'honolulu', 'permit application']
Cluster 3
['residence', 'singlefamily residence', 'smx', 'singlefamily', 'pool']
Cluster 4
['project', 'job', 'department', 'concurrence', 'effect']


In [53]:
# topic discovery with LDA

from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=5)
topics = lda.fit_transform(X)

det_df["topic"] = topics.argmax(axis=1)

terms = vectorizer.get_feature_names_out()

for i, comp in enumerate(lda.components_):
    words = np.array(terms)[np.argsort(comp)[-8:]]
    print(f"Topic {i}: {', '.join(words)}")

Topic 0: permit, lot, water, installation permit, cwrm, installation, pweng, gd
Topic 1: repairs, improveme, maui, oahu, kauai, road, project, subdivision
Topic 2: elementary school, elementary, center, job, improvements, project, residence, school
Topic 3: new, dwelling, permit, smx, county maui, maui permit, county, maui
Topic 4: coh grading, tmk, coh, grading permit, grading, application, permit application, permit
